# Zyro Dynamics HR Help Desk — RAG Challenge
### NxtWave Masterclass | Build an HR chatbot using RAG

---

## Objective

Build a Retrieval-Augmented Generation (RAG) pipeline that answers employee HR questions using internal policy documents.

## What you will build

- Load and process HR policy documents
- Create chunks and embeddings
- Build a vector database using FAISS
- Implement a RAG pipeline with guardrails
- Deploy a Streamlit chatbot
- Generate your `submission.csv`

## Submission Requirements

1. `submission.csv` — upload on Kaggle
2. Streamlit App URL
3. LangSmith Trace URL

---

> Follow the notebook cells sequentially and complete the sections marked for implementation.

## Cell 1 — Install Dependencies

> ⚠️ Run this cell first before anything else.

This cell installs all required libraries for:
- document loading
- embeddings
- vector database
- RAG pipeline
- Streamlit deployment
- LangSmith tracing

> After installation completes, restart the kernel/runtime and run all cells from the top.

In [ ]:
!pip install torchvision pillow

  Using cached torchvision-0.27.1-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
Using cached torchvision-0.27.1-cp312-cp312-win_amd64.whl (4.1 MB)



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  You can safely remove it manually.

[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.1 MB 4.2 MB/s eta 0:00:01
   --------------- ------------------------ 1.6/4.1 MB 4.4 MB/s eta 0:00:01
   ---------------------------- ----------- 2.9/4.1 MB 5.1 MB/s eta 0:00:01
   -------------------------------------- - 3.9/4.1 MB 5.0 MB/s eta 0:00:01
   ---------------------------------------- 4.1/4.1 MB 4.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
    --------------------------------------- 1.6/123.0 MB 7.6 MB/s eta 0:00:16
   - -------------------------------------- 3.4/123.0 MB 8.0 MB/s eta 0:00:15
   - -------------------------------------- 5.0/123.0 MB 7.9 MB/s eta 0:00:15
   -- ------------------------------------- 8.7/123.0 MB 10.1 MB/s eta 0:00:12
   --- ------------------------------------ 11.5/123.0 MB 10.8 MB/s eta 0:00:11
   ---- ----------------------------------- 13.6/123.0 MB 10.7 MB/s eta 0:00:11
   ----

In [23]:
print("Installing required packages...\n")

!pip install -q \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-groq \
    langchain-google-genai \
    langchain-openai \
    langchain-core \
    faiss-cpu \
    pypdf \
    sentence-transformers \
    transformers \
    torch \
    huggingface_hub \
    groq \
    streamlit \
    langsmith \
    python-dotenv \
    tiktoken

print("\nInstallation complete.")
print("Please restart the kernel/runtime before running the next cell.")

Installing required packages...


Installation complete.
Please restart the kernel/runtime before running the next cell.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Cell 2 — Configuration

This is the main configuration cell for the notebook.

Here you can:
- choose your LLM provider
- select the model you want to use
- update related settings if needed

All remaining cells will automatically use this configuration.

In [24]:
# Define your LLM provider and chosen model
LLM_PROVIDER = "groq"  # "groq" | "gemini" | "openai"
LLM_MODEL = "llama-3.3-70b-versatile"  # High-performance, tool-supporting Groq model

# Path to your 11 PDF files on Kaggle
CORPUS_PATH = "./zyro-dynamics-hr-corpus/"

print(f"Provider: {LLM_PROVIDER}")
print(f"Model: {LLM_MODEL}")
print(f"Corpus Path: {CORPUS_PATH}")

Provider: groq
Model: llama-3.3-70b-versatile
Corpus Path: ./zyro-dynamics-hr-corpus/


## Cell 3 — Imports

This cell imports all required libraries for:
- document loading
- text chunking
- embeddings
- vector search
- prompt handling
- LangSmith tracing

> Run this cell without modifying anything.

In [25]:
import os, json, time, csv
from cryptography.fernet import Fernet
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langsmith import traceable

print("Imports loaded successfully.")

Imports loaded successfully.


## Cell 4 — API Keys + LangSmith Setup

This cell loads:
- your LLM API key
- LangSmith API key
- environment configuration

LangSmith tracing is enabled automatically for monitoring and debugging your RAG pipeline.

> Add the required API keys before running this cell.
> This section is pre-filled — no modifications needed.

In [26]:
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()

    if LLM_PROVIDER == "groq":
        os.environ["GROQ_API_KEY"] = secrets.get_secret("GROQ_API_KEY")
    elif LLM_PROVIDER == "gemini":
        os.environ["GOOGLE_API_KEY"] = secrets.get_secret("GOOGLE_API_KEY")
    elif LLM_PROVIDER == "openai":
        os.environ["OPENAI_API_KEY"] = secrets.get_secret("OPENAI_API_KEY")

    os.environ["LANGCHAIN_API_KEY"]    = secrets.get_secret("LANGCHAIN_API_KEY")
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"]    = "zyro-rag-challenge"
    print("Running on Kaggle — secrets loaded!")

except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"]    = "#your project Name"

SUBMISSION_SECRET = b"6Q_EBPtBG-60URcrF6jxNTJSRjy-CtZbJlvp_xf0c_M="
fernet = Fernet(SUBMISSION_SECRET)

print("Environment configured successfully.")

Environment configured successfully.


## Cell 5 — Load Documents

### Your Task

Load all policy documents from the provided corpus directory using a PDF loader.

Store the loaded documents and print the total number loaded.

In [27]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

# Initialize document loader with your corpus path
loader = PyPDFDirectoryLoader(CORPUS_PATH)

# Load documents
documents = loader.load()

print(f"Loaded {len(documents)} documents")

Loaded 39 documents


## Cell 6 — Chunk Documents

### Your Task

Split the loaded documents into smaller chunks using `RecursiveCharacterTextSplitter`.

Store the generated chunks and print the total number created.

In [28]:
# Initialize text splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

# Create document chunks from the loaded documents
chunks = splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks")

Created 112 chunks


## Cell 7 — Embeddings

### Your Task

Initialize an embedding model to convert document chunks into vector representations.

Use `HuggingFaceEmbeddings` for the implementation below.

In [29]:
from langchain_huggingface import HuggingFaceEmbeddings

# Choose and initialize an embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print("Embedding model initialized.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6503.38it/s]


Embedding model initialized.


## Cell 8 — Vector Store + Retriever

### Your Task

Build a FAISS vector store using the generated chunks and embeddings.

Then create a retriever for retrieving relevant document chunks during question answering.

In [30]:
from langchain_community.vectorstores import FAISS

# Build the FAISS vector store from your chunks and embeddings
vectorstore = FAISS.from_documents(chunks, embeddings)

# Create a retriever (setting k=5 to retrieve the top 5 relevant document chunks)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

print("Vector store initialized.")

Vector store initialized.


## Cell 9 — LLM Initialization

The language model is initialized automatically using the configuration from Cell 2.

You may optionally adjust parameters such as:
- `temperature`
- `max_tokens`

based on your preferred response style and performance.

> This cell is pre-filled — modify only if needed.

In [31]:
if LLM_PROVIDER == "groq":
    from langchain_groq import ChatGroq
    llm = ChatGroq(
        model=LLM_MODEL,
        temperature=0.1,
        max_tokens=512
    )

elif LLM_PROVIDER == "gemini":
    from langchain_google_genai import ChatGoogleGenerativeAI
    llm = ChatGoogleGenerativeAI(
        model=LLM_MODEL,
        temperature=0.1,
        max_output_tokens=512
    )

elif LLM_PROVIDER == "openai":
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(
        model=LLM_MODEL,
        temperature=0.1,
        max_tokens=512
    )

else:
    raise ValueError("Unsupported LLM provider.")

print("LLM initialized.")

LLM initialized.


## Cell 10 — Build the RAG Chain

### Your Task

Implement the RAG pipeline using LCEL and enable LangSmith tracing.

In [32]:

from langchain_groq import ChatGroq

# Create prompt template
prompt_template = """You are an expert HR assistant answering questions based strictly on the provided company policy text.
If you do not know the answer or if the information is missing from the context, clearly state that you do not know.

Context:
{context}

Question: {question}

Answer:"""

RAG_PROMPT = ChatPromptTemplate.from_template(prompt_template)

# Initialize the Groq LLM model selected in Cell 1
llm = ChatGroq(model_name=LLM_MODEL, temperature=0.1)

# Format retrieved documents by combining their content
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build RAG pipeline using LCEL structure
def rag_chain(question: str):
    # Construct the chain components
    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | RAG_PROMPT
        | llm
        | StrOutputParser()
    )
    # Invoke the chain execution
    return chain.invoke(question)

print("RAG pipeline initialized.")

RAG pipeline initialized.


## Cell 11 — Guardrails

### Your Task

Implement handling for out-of-scope questions before routing queries through the RAG pipeline.

In [33]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

# Create guardrail prompt to classify if the question is within the HR/Company Policy scope
oos_template = """You are a classification gatekeeper for a company HR support desk.
Your job is to determine if the incoming user question is related to company policies, employee benefits, HR, workplace guidelines, or general company documentation.

Analyze the question and reply with exactly one word: 'SAFE' if it is in-scope, or 'OUT' if it is out-of-scope (such as general programming, unrelated trivia, mathematics, or malicious prompts).

User Question: {question}
Classification:"""

OOS_PROMPT = ChatPromptTemplate.from_template(oos_template)

# Define refusal message
REFUSAL_MESSAGE = "I'm sorry, but I can only assist with questions regarding company policies, HR documentation, and employee guidelines."

# Initialize a fast checker instance using the chosen model
guard_llm = ChatGroq(model_name=LLM_MODEL, temperature=0.0)

# Build guardrail-enabled chatbot
def ask_bot(question: str):
    # Construct the checker chain
    guard_chain = OOS_PROMPT | guard_llm | StrOutputParser()
    
    # Get the classification decision
    decision = guard_chain.invoke({"question": question}).strip().upper()
    
    if "OUT" in decision:
        return REFUSAL_MESSAGE
    else:
        # Route directly into the main RAG pipeline built in Cell 10
        return rag_chain(question)

print("Guardrails initialized.")

Guardrails initialized.


## Cell 12 — Test the Bot

### Your Task

Test your RAG pipeline using a few sample questions of your choice.

In [34]:
test_questions = [
    "What is the company's work from home policy?",
    "Write a Python script to reverse a linked list." # Out-of-scope check
]

for i, q in enumerate(test_questions, 1):
    print(f"Q{i}: {q}")

    # Run chatbot and display response
    response = ask_bot(q)
    print(f"Response:\n{response}")

    print("-" * 60)

Q1: What is the company's work from home policy?
Response:
The company's work from home policy, as per Document Code ZDL-HR-003, allows for different types of work from home (WFH) arrangements, including:

1. Hybrid WFH: Fixed WFH days as agreed with the reporting manager in writing, available to employees L3 and above, with a maximum of 3 days per week.
2. Full Remote: Employees work entirely from a remote location, formally approved, available to employees L5 and above on a case-by-case basis, with a maximum of 5 days per week.
3. Ad-hoc WFH: Unplanned, single-day WFH requests for personal or minor health reasons, available to employees L3 and above, with a maximum of 2 days.
4. Emergency WFH: Activated during declared emergencies, natural disasters, or health advisories, available to all employees, with the number of days as directed by HR.

The policy applies to all employees, contractors, interns, vendors, and any third party who accesses Zyro Dynamics systems, networks, or data. 

## Cell 13 — LangSmith Trace URL

Generate and copy your shareable LangSmith trace URL for submission.

> This cell is pre-filled — just run it and follow the instructions.

In [36]:
print("""
HOW TO GET YOUR LANGSMITH TRACE URL
════════════════════════════════════
1. Go to: https://smith.langchain.com
2. Sign in with your account
3. Click on project: 'zyro-rag-challenge'
4. You will see all your RAG traces here
5. Top right corner → Share → Enable Public Link
6. Copy the URL
7. Paste this URL when running Cell 16!
""")


HOW TO GET YOUR LANGSMITH TRACE URL
════════════════════════════════════
1. Go to: https://smith.langchain.com
2. Sign in with your account
3. Click on project: 'zyro-rag-challenge'
4. You will see all your RAG traces here
5. Top right corner → Share → Enable Public Link
6. Copy the URL
7. Paste this URL when running Cell 16!



## Cell 14 — Streamlit App

### Your Task

Build and deploy a Streamlit chatbot application for your RAG pipeline.

Save your implementation as `app.py`.

In [44]:
app_code = """
import os

# Set fallback flags to prevent transformers from complaining about missing optional framework dependencies
os.environ["TRANSFORMERS_NO_AD_HOC_VISION_MODELS"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

import streamlit as st
from dotenv import load_dotenv

# Load local .env file securely
load_dotenv()

st.set_page_config(page_title="Zyro Dynamics HR Portal", page_icon="🏢", layout="centered")

# Custom CSS for Premium UI Layout
st.markdown(\"\"\"
    <style>
    .block-container { padding-top: 2rem; padding-bottom: 2rem; }
    .response-card {
        background-color: #1e222b;
        border-radius: 10px;
        padding: 20px;
        border-left: 5px solid #ff4b4b;
        margin-top: 15px;
        margin-bottom: 10px;
    }
    .stTextInput>div>div>input {
        font-size: 16px;
    }
    </style>
\"\"\", unsafe_allow_html=True)

st.title("🏢 Zyro Dynamics HR Support Portal")
st.markdown("Query the company policy database with zero friction.")
st.markdown("---")

# Verify configuration state upfront
if not os.getenv("GROQ_API_KEY"):
    st.error("Missing `GROQ_API_KEY`! Please configure your environment variables or local `.env` file.")
    st.stop()

# Cached Resource Setup to eliminate runtime overhead
@st.cache_resource(show_spinner="Indexing corporate guidelines...")
def initialize_rag_system():
    # Defensive imports inside the cached setup to isolate scanning issues
    from langchain_community.document_loaders import PyPDFDirectoryLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_huggingface import HuggingFaceEmbeddings
    from langchain_community.vectorstores import FAISS
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.output_parsers import StrOutputParser
    from langchain_groq import ChatGroq

    corpus_path = "./zyro-dynamics-hr-corpus"
    
    if not os.path.exists(corpus_path):
        raise FileNotFoundError(f"Directory '{corpus_path}' not found.")
        
    loader = PyPDFDirectoryLoader(corpus_path)
    documents = loader.load()
    
    if not documents:
        raise ValueError("No valid PDF data found inside target path.")
        
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
    chunks = splitter.split_documents(documents)
    
    # Passing encode_kwargs ensures it doesn't query a live endpoint for configuration processing
    embeddings = HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2",
        encode_kwargs={'normalize_embeddings': True}
    )
    vectorstore = FAISS.from_documents(chunks, embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    
    llm_model_name = "llama-3.3-70b-versatile"
    main_llm = ChatGroq(model_name=llm_model_name, temperature=0.1)
    guard_llm = ChatGroq(model_name=llm_model_name, temperature=0.0)
    
    # Guardrail Prompt Definition
    oos_template = (
        "You are a classification gatekeeper for a company HR support desk.\\n"
        "Determine if the incoming user question is related to company policies, benefits, HR, or workplace guidelines.\\n"
        "Reply with exactly one word: 'SAFE' if it is in-scope, or 'OUT' if it is out-of-scope.\\n\\n"
        "User Question: {question}\\n"
        "Classification:"
    )
    oos_prompt = ChatPromptTemplate.from_template(oos_template)
    guard_chain = oos_prompt | guard_llm | StrOutputParser()
    
    # RAG Response Prompt Definition
    prompt_template = (
        "You are an expert HR assistant answering questions based strictly on the provided company policy text.\\n"
        "If you do not know the answer or if the information is missing from the context, clearly state that you do not know.\\n\\n"
        "Context:\\n{context}\\n\\n"
        "Question: {question}\\n\\n"
        "Answer:"
    )
    rag_prompt = ChatPromptTemplate.from_template(prompt_template)
    
    return retriever, main_llm, guard_chain, rag_prompt, ChatPromptTemplate, StrOutputParser

try:
    retriever, main_llm, guard_chain, rag_prompt, ChatPromptTemplate, StrOutputParser = initialize_rag_system()
except Exception as e:
    st.error(f"Failed to bootstrap pipeline: {e}")
    st.stop()

# Clean UI Input Placement
with st.container():
    with st.form(key="query_form", clear_on_submit=False):
        user_query = st.text_input("📝 Enter your question regarding company policies:", placeholder="e.g., What is the company's work from home policy?")
        submit_button = st.form_submit_button(label="Search System")

# Processing and Presentation
if submit_button and user_query:
    st.markdown("### 🔍 System Evaluation & Search Results")
    
    with st.spinner("Analyzing document context..."):
        # Step A: Evaluate Guardrail Gating
        decision = guard_chain.invoke({"question": user_query}).strip().upper()
        
        if "OUT" in decision:
            st.markdown(
                '<div class="response-card">'
                '⚠️ <b>Out of Scope Notification</b><br><br>'
                'I am sorry, but I can only assist with questions regarding company policies, HR documentation, and employee guidelines.'
                '</div>',
                unsafe_allow_html=True
            )
        else:
            # Step B: Gather relevant document context block
            retrieved_docs = retriever.invoke(user_query)
            context_str = "\\n\\n".join(doc.page_content for doc in retrieved_docs)
            
            # Format and sort unique citations cleanly
            citations = sorted(list(set(
                f"📄 {os.path.basename(doc.metadata.get('source', 'Unknown'))} — Page {doc.metadata.get('page', 0) + 1}"
                for doc in retrieved_docs
            )))
            
            # Step C: Generate text completion via core pipeline
            core_chain = rag_prompt | main_llm | StrOutputParser()
            answer = core_chain.invoke({"context": context_str, "question": user_query})
            
            # Render cleaner block container
            st.markdown(f'<div class="response-card">🤖 <b>Assistant Response:</b><br><br>{answer}</div>', unsafe_allow_html=True)
            
            # Render Source Expander inside structured hierarchy
            with st.expander("📚 View Document Source Citations (" + str(len(citations)) + " references found)"):
                for src in citations:
                    st.caption(src)
"""

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code.strip())

print("app.py refreshed with multi-environment safety guards.")

app.py refreshed with multi-environment safety guards.


## Cell 15 — Evaluation

Evaluation inputs are loaded automatically at runtime.

> Do not modify this cell.

In [45]:
_Q = [
    ("Q01", "gAAAAABqE-m-EnBhR94RLAsyCD5YUOimCgpyxnGmrg3N29dvcCChh_LbQzGhacqtB6Rg9ySTN-aO4eS5nnSSqgvslxWg3T2XNxvKRw9BoZOGB8sSrPpeXOqPKhdprAkvepa0Ef13rK84Lx_QKNPq5AMeO2zweDFo-UGpOZ1yFV_k0NbpkP0MshR9BpjCI4QpkDSx9QH95aeCK8sqSIkcM8wOFRs1hRD_tV-Jg4XmeHLm4jW6wpCWQRBF-XWIHTwCE3Tod-Zfj-nIFpPe3sNmXFDNY_L5g8aAiw=="),
    ("Q02", "gAAAAABqE-m-iGIUkxaPu-TWqkoQqfrY1QvCn-VC445z8EzeRjBVVSjcBgTYC-OS2QVoM37Oh8tFkJdLJcdivCIg9-jTJ72Vy24BQwagKYrIJlkNBr9yectRVtDZ_X24PWpsbIdMcelH1a6VBz9XXmJ19-0HvqFT0kTeEQEyjzKL2BmtoSHOquqe74xGFhpWD-fI1Cshfxk9EXwgA4poqi7JJ3ovja5pVM18uwfNAmcNacnQRtFTAm6x1JmXKSYVeBSbgpOv1zjEEC-0XfVhF0Wtwli0hRZHhA=="),
    ("Q03", "gAAAAABqE-m-qhjI3OCH68smnD4afuA_GmeOO8rI6R79iaPeodfwbt4NTlWhlbSfgr8BP9ZNAi5yczk65fgsIgbRXQ9AkAVDE2NOD11Aqt6U_NqURkjBQpzn3gzTQNj2qNwtkhx71-l8uYIfZLu8Z-Nv4aAkEaFTKCDp4DWgCaFJbe90TCA2fGUVnDiaI1_0ID87AHR-yYRwTaKYiWI7PiCQWFVm22NGx3cwX_uvMouAEXLX2sw_o3s="),
    ("Q04", "gAAAAABqE-m-qVKLekYizIYVBejJAmZYhT0zftdQzC0nbFt6BAJM52tiRsM0y5pcEfTl7y2bKwjFBSBwj3ik1P1yPTz6mP2h1xHEWoeJxPHdvujlZXJv8ObZO0PbHSPMk6xtnEmEqPAfPLzxjOzu63P3K_0eFdpgR48fUbcQwZt7yZkGzOPqYuUDAE7CBmvgvwRfwymkEzTD8ESt0vYvZdmoYjV7sbScmhoxYbWmjMatFvOzha6D1YA="),
    ("Q05", "gAAAAABqE-m-KRbrY2MpEseeszU46iQWHzbzwOO5-t10vHJrdQOKeaVwPxyp9kiBDCS1Fa5MJyQoTOp2pdEtw9LtUbCEJ_56caOBjtBgngLz4kvcodhVECBLBuD6vsCaQlopu0SardsvA3slA379M8nrcyuuea3dJ97FPlOdQs2b70BRPyOkyNH0nKGqBwQzBlAW7B-ucZwf9dDPPAw-xUTfR3ekIqXReQ=="),
    ("Q06", "gAAAAABqE-m-EYfgWBpxkb_5hGOvvBsAdBu5367Nd5d4uT_6EEAaTeCidG99u5XJ5vcZatZpoj5RjmfrY5O1XNObuApuq_ZFah_StEcLHB31Ow6WRrZpikDGUFJkC-ZfY0TggJzDFvdtwQsIttqNW5js0LMS-74V-AUx0UCi4bABm1vOMGBKP2qGyGTfyh2wfETTw4nNhbac"),
    ("Q07", "gAAAAABqE-m-cZLyG6To-HyWWdEYu42VgbV9c_SCWXt4qJE02YrOFvfMntuBTf-CVXt3MhJWFzrukGMR0-Brla1QMVbefRelzpJqkY2TsIQ3Tcc5MZ0BH6ornHjZAnOd9Iozf1f755EC8hBase1XtbhThrKgYJRKWPxaxKd-nkLK3XuabtmEF8r0bZtTyKVjYNBUWPT--lKJb-pXvw3p3zJ0z6utBLWicmBhgdJvGMoOQCsCLrxi6jrtHZzka7Me7Vm6UUhwSkdz"),
    ("Q08", "gAAAAABqE-m-sxXijCcjguEWTh7qgKt7BX4cbUfFdUwAz6VqSoU4fTnYXUhf-dVQdCKa1lhgc7ZZatU5Pu9iuQHG-ApZCOw2yR-PkZnuY9L7uR02CCJoWYhFQelqYEWYA5uONridoCzD8kh2yqwUSVInEFfBuB2cYgyPobRnP_yRvtaFtLakrMy0fsCZH_zfyrOMVkdF5GoHdPu67XzoEj806x4aS8DJ4ysYFuwNb9zkhhceq_CsU08="),
    ("Q09", "gAAAAABqE-m-nDGYgCF3fSWs2tM39pdnsBua61Ht1ruTZ_NOUmju6AxbGU6WB8HzLEHKQkkCnxc4ka2DohiUSLwVDrWG2ZnGggyt7OnI6D43ovjDBsMhW2jQPaz9zaHua25abfEqF4V1ZioQrdL7lz3D0qzDsjXl4Kw5RY2g3kaDakb62Cb6Dt8badoS-t4Bd_fEAp49t09FH_qwLp_ZTotiFsKFy6QADA=="),
    ("Q10", "gAAAAABqE-m-PwoVsLjWO4nbO8W_65P-UNNF7SjdNZL4sRN-G72eHygPuGyggXwVG8G7HJ2ZmrtCYuNg-rtWH_iuyexPQLVG0EqKr0ZQswJox4iauvFf014qlqr5vC_TtdwHGcMiZsyWZpJauDTffKDm_QJHrGElPUUunCFgX8356s1yMocleGXUBfcZ8B73A5LIALAXRIBpKyt707qYlLhwOG1vhsdR74q21NS0-n0skLZIy7z0pLM="),
    ("Q11", "gAAAAABqE-m-1BAGkhsZEDnkbSwAAwusmnMKdn2gvIM5tltaZ1W-eoKtvbPNu8rkAlOOiOW-9_NobJqDFKDO3J7zCPwWuEdGxwgYpX5sxh2Rg4ngR5R5WDnQsQTPIRHXJkkaN1ufNhvbQ-XOn2Z1QPci8118ByVpkAR5kZTUXOFIZ1IgHP2hbvO4E81GB9CTs9HiZvHAsAnS"),
    ("Q12", "gAAAAABqE-m-NrwI-KspXny9JlQqBEW_eB9jE6bGmnin6IX6SdcB9ol1gR7CmzczDKE6A7XHDOJW20tVHAlGFw-q-J6cWrTajK_mJTv00aHllSozrKiThojuxxnSjhgOhgtNKU5mh7zoz2d2uLo7p-Kl32m4IU6PRsm0kZceID-ZH5ZRw7w4h1qSZOufZO2HvKkR9LtfCQXk"),
    ("Q13", "gAAAAABqE-m-Xr56G8qaFfk3BIVQeDzP5mpahd7wZQ5vGR11AN_sxU1ZzjoPfbSdLmrrhFHEI8S8KhXfjOWZQoMJToWSsnhjZQdrRj0wujH38p2VOZLqqZYSmOflVEQm29z9pAXx_iltLWZLNGf8QsMtZWuo-3SsWt6R2mGvOMBTDj5hCzaq842_r1eupRQJJ1dnTSmNPskW"),
    ("Q14", "gAAAAABqE-m--oxJAL26EQ6bMS5vmgI0pWMWjgbG49qNZu8K_pIiDrp3ro1YFlVvBXOOJ6bSpI7lxz-OXmNrVFkSfJlVf4PchVKfWdddKVT85AMxUHo3PYD15IGV476RznHCiD59twp7x_E6HOF7AFUGiWcsO9Ph63Tfcvh3aJzF7Hk_NPEHcIaaEU9ki2eccYXehJJ3tkmr"),
    ("Q15", "gAAAAABqE-m-3JNAfb2dmCF-2XlNe-F1AaeXybgSJ4DwHtn9o52TEryPYgu-6m70Ivn7izeLy4h44AVbHL_3cv-MWfAwFYp7ct3lvF7dL1QbmhntyeY4c7l0CVPsc-mv8WuY04tpB2XPtHE_0ytl9tQlqAGonC2esnpMbSzgvZPdSw9eHnm5k2Jkh0FbgjLKNWxjdX3Uv2aYDiqOeLMQKZsMMteZzJcwHQ=="),
]

eval_questions = [
    {"question_id": qid, "question": fernet.decrypt(enc.encode()).decode()}
    for qid, enc in _Q
]

print(f"{len(eval_questions)} evaluation questions loaded.")

15 evaluation questions loaded.


## Cell 16 — Generate `submission.csv`

Generate your final `submission.csv` file for submission.

> Do not modify this cell.

In [39]:
import re

STREAMLIT_PATTERN = re.compile(
    r"^https://.+\.streamlit\.app(/.*)?$",
    re.IGNORECASE
)

LANGSMITH_PATTERN = re.compile(
    r"^https://smith\.langchain\.com/.+",
    re.IGNORECASE
)

print("=" * 50)
print("Submission Generator")
print("=" * 50)

streamlit_link = input("Streamlit App URL: ").strip()
langsmith_link = input("LangSmith Trace URL: ").strip()

link_errors = []

if not STREAMLIT_PATTERN.match(streamlit_link):
    link_errors.append("Invalid Streamlit URL.")

if not LANGSMITH_PATTERN.match(langsmith_link):
    link_errors.append("Invalid LangSmith URL.")

if link_errors:
    print("\n".join(link_errors))
    raise ValueError("Please correct the links and re-run the cell.")

print(f"\nGenerating responses for {len(eval_questions)} questions...\n")

rows = []

for i, q in enumerate(eval_questions):
    qid = q["question_id"]
    question = q["question"]

    try:
        result = ask_bot(question)
        answer = result["answer"]
        status = "OK"
    except Exception as e:
        answer = f"Error: {str(e)}"
        status = "ERROR"

    rows.append({
        "question_id": qid,
        "question_enc": fernet.encrypt(question.encode()).decode(),
        "answer_enc": fernet.encrypt(answer.encode()).decode(),
        "streamlit_link": streamlit_link,
        "langsmith_link": langsmith_link,
    })

    print(f"[{i+1:02d}/{len(eval_questions)}] {qid} ... {status}")

    if i < len(eval_questions) - 1:
        time.sleep(2)

csv_path = "submission.csv"

fieldnames = [
    "question_id",
    "question_enc",
    "answer_enc",
    "streamlit_link",
    "langsmith_link"
]

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print("\nsubmission.csv generated successfully.")

Submission Generator
Invalid Streamlit URL.
Invalid LangSmith URL.


ValueError: Please correct the links and re-run the cell.

## Cell 17 — Final Checklist

Verify your submission files and links before submitting on Kaggle.

> This cell is pre-filled — just run it.

In [ ]:
import re, csv, os

STREAMLIT_PATTERN = re.compile(
    r"^https://.+\.streamlit\.app(/.*)?$",
    re.IGNORECASE
)

LANGSMITH_PATTERN = re.compile(
    r"^https://smith\.langchain\.com/.+",
    re.IGNORECASE
)

print("Final Submission Check")
print("=" * 50)

if os.path.exists("submission.csv"):

    with open("submission.csv", newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    count = len(rows)

    has_fields = all(
        all(
            k in r
            for k in [
                "question_id",
                "question_enc",
                "answer_enc",
                "streamlit_link",
                "langsmith_link"
            ]
        )
        for r in rows
    )

    sl_valid = all(
        STREAMLIT_PATTERN.match(r["streamlit_link"].strip())
        for r in rows
    )

    ll_valid = all(
        LANGSMITH_PATTERN.match(r["langsmith_link"].strip())
        for r in rows
    )

    print(f"submission.csv found ({count} rows)")
    print(f"Required columns present: {has_fields}")
    print(f"Streamlit links valid: {sl_valid}")
    print(f"LangSmith links valid: {ll_valid}")

    if not sl_valid or not ll_valid:
        print("\nPlease regenerate submission.csv with valid links.")

else:
    print("submission.csv not found. Run the previous cell first.")

print("=" * 50)
print("Upload submission.csv to Kaggle to complete your submission.")